# Capítulo 1 — Introdução

Códigos do livro *Backpropagation e Redes Neurais* (Araken de Medeiros Santos).

> **Aviso:** este notebook foi gerado a partir dos scripts do livro e **não foi executado como notebook**; as saídas reportadas no texto vêm da execução dos scripts `.py` correspondentes. Execute as células na ordem, a partir desta pasta, com o ambiente de `requirements.txt` (ver Apêndice E). Valores podem diferir ligeiramente em outras plataformas.

In [ ]:
import os, sys
RAIZ = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(RAIZ, 'codigo'))
os.environ.setdefault('FIGS', os.path.join(RAIZ, 'figs'))
print('rnbook em', os.path.join(RAIZ, 'codigo'))

## Script `cap01/exp_primeira_rede.py`

In [ ]:
"""Cap. 1 - 'Olá, rede': primeira rede treinada com a infraestrutura do livro.
Ajusta y = sen(2*pi*x) com ruído, MLP 1-8-1 (tanh), treinada por Levenberg-Marquardt
(implementação didática do livro). Semente fixa; imprime números citados no texto."""
import sys, os
import numpy as np
from rnbook import MLP
from rnbook import otimizadores as O
from rnbook.estilo import plt, COR, salvar

rng = np.random.default_rng(2026)
N = 40
x = np.sort(rng.uniform(0, 1, N))[:, None]
t = np.sin(2 * np.pi * x) + rng.normal(0, 0.15, size=x.shape)

rede = MLP([1, 8, 1], ["tanh", "identidade"], semente=7)
print("parâmetros:", rede.n_params, "(= 8*1+8 + 1*8+1)")
E0 = rede.perda(rede.theta, x, t)
res = lambda th: rede.residuos(th, x, t)
jac = lambda th: rede.jacobiana(th, x)
r = O.levenberg_marquardt(rede.theta, res, jac, max_iter=60)
th = r.theta
E = rede.perda(th, x, t)
print(f"perda inicial (sse) = {E0:.4f}; perda final = {E:.4f}; iterações = {r.n_iter}; parada = {r.motivo}")
rmse = np.sqrt(np.mean((rede.prever(x, th) - t) ** 2))
print(f"RMSE no treino = {rmse:.4f} (ruído verdadeiro: 0.15)")

xx = np.linspace(0, 1, 300)[:, None]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(x, t, s=22, color=COR["gry"], label="dados com ruído", zorder=3)
ax[0].plot(xx, np.sin(2 * np.pi * xx), "--", color=COR["gry"], label="função verdadeira")
ax[0].plot(xx, rede.prever(xx, rede.theta), color=COR["red"], alpha=.8, label="rede antes do treino")
ax[0].plot(xx, rede.prever(xx, th), color=COR["ind"], lw=2.2, label="rede depois do treino")
ax[0].set_xlabel("x"); ax[0].set_ylabel("y"); ax[0].legend(); ax[0].set_title("Aprendendo uma curva")
ax[1].semilogy(r.historico["perda"], color=COR["ind"], lw=2)
ax[1].set_xlabel("iteração"); ax[1].set_ylabel("perda E (escala log)"); ax[1].set_title("A perda cai a cada iteração")
salvar(fig, "f1_primeira_rede")

## Script `cap01/figs_cap01.py`

In [ ]:
"""Figuras conceituais do Capítulo 1 (todas geradas por código; nada de imagens prontas)."""
import sys, os
import numpy as np
from rnbook.estilo import plt, COR, salvar
from rnbook.desenho import caixa, seta, diagrama_rede

# ---- f1_camadas: cinco conceitos distintos
fig, ax = plt.subplots(figsize=(10, 3.9)); ax.set_xlim(0, 10); ax.set_ylim(0, 3.9); ax.axis("off"); ax.grid(False)
itens = [("Neurônio\nartificial", "$z=\\mathbf{w}^T\\mathbf{x}+b$\n$a=\\phi(z)$", COR["cya"]),
         ("Arquitetura\nda rede", "camadas, neurônios,\nconexões", COR["ind"]),
         ("Função\nde perda", "$E(\\theta)$: mede o erro\nda rede", COR["amb"]),
         ("Algoritmo de\ndiferenciação", "backpropagation:\ncalcula $\\nabla E$", COR["red"]),
         ("Método de\notimização", "usa $\\nabla E$ para\natualizar $\\theta$", COR["grn"])]
w = 1.7
for i, (t, d, c) in enumerate(itens):
    x = 0.15 + i * 1.97
    caixa(ax, x, 1.55, w, 1.6, t, cor=c, fc="white", fs=10.5, bold=True)
    ax.text(x + w / 2, 0.95, d, ha="center", va="center", fontsize=9, color=COR["ink"])
    ax.add_patch(plt.Circle((x + w / 2, 3.45), 0.2, color=c)); ax.text(x + w / 2, 3.45, str(i + 1), color="white", ha="center", va="center", fontweight="bold")
    if i < 4: seta(ax, (x + w + 0.02, 2.35), (x + 1.97 - 0.02, 2.35), cor=COR["gry"])
ax.text(5, 0.2, "Cinco ideias diferentes. Misturá-las é uma das maiores fontes de confusão para quem está começando.", ha="center", fontsize=9.5, style="italic", color=COR["gry"])
salvar(fig, "f1_camadas")

# ---- f1_neuronio: analogia biológica x artificial
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
a = ax[0]; a.set_xlim(0, 10); a.set_ylim(0, 6); a.axis("off"); a.grid(False); a.set_title("Neurônio biológico (esquema)"); a.set_aspect("equal")
a.add_patch(plt.Circle((5, 3), 1.0, fc="#FDE68A", ec=COR["amb"], lw=2)); a.text(5, 3, "corpo\ncelular", ha="center", va="center", fontsize=9)
for ang, (dx, dy) in zip(range(4), [(-3.5, 2), (-3.5, 0.5), (-3.5, -0.5), (-3.2, -2)]):
    a.plot([5 - 1, 5 + dx * 0.55 + 0.5], [3, 3 + dy * 0.7], color=COR["gry"], lw=2)
    a.plot([5 + dx * 0.55 + 0.5, 5 + dx * 0.9 + 0.5], [3 + dy * 0.7, 3 + dy * 1.0], color=COR["gry"], lw=1.2)
a.text(1.1, 5.2, "dendritos\n(recebem sinais)", fontsize=9, ha="center"); a.plot([6, 9.4], [3, 3], color=COR["gry"], lw=3); a.text(7.8, 3.5, "axônio\n(envia o sinal)", ha="center", fontsize=9)
b = ax[1]; b.set_xlim(0, 10); b.set_ylim(0, 6); b.axis("off"); b.grid(False); b.set_title("Neurônio artificial"); b.set_aspect("equal")
ys = [4.6, 3.4, 2.2, 1.0]
for i, y in enumerate(ys):
    b.text(0.4, y, f"$x_{i+1}$", ha="center", va="center", fontsize=11); seta(b, (0.8, y), (4.0, 3.0), cor=COR["gry"])
    b.text(2.2, 3.0 + (y - 3.0) * 0.45 + 0.28, f"$w_{i+1}$", fontsize=10, color=COR["ind"])
b.add_patch(plt.Circle((4.8, 3), 0.85, fc=COR["ind"], ec="white")); b.text(4.8, 3.0, "$\\Sigma$ ", color="white", fontsize=15, ha="center", va="center")
b.text(4.8, 4.35, "$b$ (viés)", ha="center", fontsize=10, color=COR["amb"]); seta(b, (4.8, 4.2), (4.8, 3.9), cor=COR["amb"])
b.add_patch(plt.Circle((7.0, 3), 0.7, fc=COR["grn"], ec="white")); b.text(7.0, 3.0, "$\\phi$", color="white", fontsize=15, ha="center", va="center")
seta(b, (5.7, 3), (6.2, 3)); seta(b, (7.75, 3), (9.4, 3)); b.text(9.6, 3, "$a$", fontsize=12, va="center")
b.text(4.8, 1.75, "$z=\\sum w_i x_i + b$", ha="center", fontsize=9.5); b.text(7.0, 1.75, "$a=\\phi(z)$", ha="center", fontsize=9.5)
salvar(fig, "f1_neuronio")

# ---- f1_rede_dim: rede 3-4-2 com dimensões
fig, ax = plt.subplots(figsize=(8.4, 4.0))
diagrama_rede(ax, [3, 4, 2], rotulos=[["x₁", "x₂", "x₃"], ["a₁", "a₂", "a₃", "a₄"], ["y₁", "y₂"]],
              titulos=["entrada\n$n_0=3$", "camada oculta\n$n_1=4$", "saída\n$n_2=2$"], passo_x=2.6, raio=0.17)
ax.text(1.3, -1.55, "$W^{(1)}$: 4×3,  $b^{(1)}$: 4", ha="center", fontsize=10, color=COR["ind"])
ax.text(3.9, -1.55, "$W^{(2)}$: 2×4,  $b^{(2)}$: 2", ha="center", fontsize=10, color=COR["ind"])
ax.text(2.6, -2.05, "total de parâmetros: $(4\\cdot3+4)+(2\\cdot4+2)=26$", ha="center", fontsize=10, color=COR["ink"])
ax.set_ylim(-2.4, 2.2)
salvar(fig, "f1_rede_dim")

# ---- f1_linha_tempo
eventos = [(1943, "McCulloch–Pitts\nneurônio lógico", COR["cya"], 1), (1949, "Hebb\nregra de\naprendizado", COR["gry"], -1), (1958, "Rosenblatt\nperceptron", COR["ind"], 1),
           (1960, "Widrow–Hoff\nADALINE / LMS", COR["grn"], -1), (1969, "Minsky–Papert\nlimites do\nperceptron", COR["red"], 1), (1974, "Werbos\n(tese)", COR["amb"], -1),
           (1986, "Rumelhart,\nHinton, Williams\nbackpropagation", COR["ind"], 1), (1988, "Fahlman: Quickprop\nJacobs: Delta-Bar-Delta", COR["vio"], -1),
           (1989, "Cybenko; Hornik et al.\naproximação universal", COR["grn"], 1), (1993, "Riedmiller–Braun\nRPROP", COR["pnk"], -1),
           (1994, "Hagan–Menhaj\nLM em redes", COR["amb"], 1), (2003, "Igel–Hüsken\niRPROP", COR["cya"], -1)]
fig, ax = plt.subplots(figsize=(11, 4.4)); ax.axhline(0, color=COR["ink"], lw=2); ax.axis("off"); ax.grid(False)
anos = np.array([e[0] for e in eventos]); xs = np.interp(np.arange(len(eventos)), [0, len(eventos) - 1], [0, 1]) * 22
for (ano, txt, c, lado), x in zip(eventos, xs):
    ax.plot([x, x], [0, 0.55 * lado], color=c, lw=1.5); ax.scatter([x], [0], s=60, color=c, zorder=3)
    ax.text(x, 0.62 * lado, txt, ha="center", va="bottom" if lado > 0 else "top", fontsize=7.8, color=COR["ink"])
    ax.text(x, -0.17 * lado if lado > 0 else 0.17, str(ano), ha="center", va="center", fontsize=8.5, fontweight="bold", color=c)
ax.set_xlim(-1.2, 23.2); ax.set_ylim(-1.5, 1.7)
ax.set_title("Alguns marcos da história que levam aos algoritmos deste livro (espaçamento não proporcional ao tempo)", fontsize=10)
salvar(fig, "f1_linha_tempo")

# ---- f1_mapa: mapa do livro
fig, ax = plt.subplots(figsize=(10, 4.2)); ax.set_xlim(0, 10); ax.set_ylim(0, 4.2); ax.axis("off"); ax.grid(False)
caixa(ax, 0.1, 3.0, 4.6, 0.9, "PARTE 1 — Fundamentos\nCap. 1 Introdução · Cap. 2 Matemática e Python", cor=COR["ind"], fs=9.5, bold=True)
caixa(ax, 5.3, 3.0, 4.6, 0.9, "PARTE 2 — Treinamento de redes\nPerceptron → Backpropagation → 4 famílias de algoritmos", cor=COR["cya"], fs=9.5, bold=True)
caps = [("3", "Perceptron", COR["ind"]), ("4", "Backprop\ne MLP", COR["red"]), ("5", "QPROP", COR["amb"]), ("6", "RPROP\n+ 3 variantes", COR["grn"]), ("7", "Levenberg–\nMarquardt", COR["vio"]), ("8", "DBD e EDBD", COR["pnk"])]
for i, (n, t, c) in enumerate(caps):
    x = 0.15 + i * 1.64
    caixa(ax, x, 1.1, 1.45, 1.2, f"Cap. {n}\n{t}", cor=c, fc="white", fs=9, bold=True)
    if i < 5: seta(ax, (x + 1.47, 1.7), (x + 1.62, 1.7), cor=COR["gry"], ms=9)
ax.text(5, 0.55, "Do neurônio isolado ao treinamento eficiente: cada capítulo acrescenta uma ideia sobre o anterior", ha="center", fontsize=10, style="italic", color=COR["gry"])
ax.text(2.6, 0.1, "gradiente, derivada, Newton", ha="center", fontsize=8.5, color=COR["gry"]); ax.text(7.8, 0.1, "gradientes locais → passos adaptativos → curvatura", ha="center", fontsize=8.5, color=COR["gry"])
salvar(fig, "f1_mapa")
print("figuras do cap. 1 ok")

## Exercícios práticos e projeto: soluções de referência

### 1

Neste exercício você treinará uma rede do scikit-learn para aprender a curva $y=\sin(2\pi x)$ com ruído e verificará se ela **generaliza**.

1. Gere 200 pontos $x\in[0,1]$ e $y=\sin(2\pi x)+\varepsilon$, com $\varepsilon\sim\mathcal{N}(0;\,0{,}15^2)$ (semente 0).
2. Separe 70% para treino e 30% para teste (`train_test_split`, `random_state=0`).
3. Treine `MLPRegressor` com uma camada oculta de 8 neurônios, ativação `tanh` e `solver="lbfgs"`.
4. Calcule o $R^2$ no treino e no teste e conte os parâmetros da rede.

**Pergunta:** a rede generalizou? O número de parâmetros confere com a fórmula do Capítulo 1?

In [ ]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

rng = np.random.default_rng(0)
x = rng.uniform(0, 1, 200)[:, None]
t = np.sin(2 * np.pi * x).ravel() + rng.normal(0, 0.15, 200)
x_tr, x_te, t_tr, t_te = train_test_split(x, t, test_size=0.3, random_state=0)

modelo = MLPRegressor(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs",
                      max_iter=2000, random_state=0).fit(x_tr, t_tr)
print("R2 treino = %.4f | R2 teste = %.4f" % (r2_score(t_tr, modelo.predict(x_tr)),
                                              r2_score(t_te, modelo.predict(x_te))))
P = sum(w.size for w in modelo.coefs_) + sum(b.size for b in modelo.intercepts_)
print("parâmetros:", P, "| fórmula 1-8-1:", (8 * 1 + 8) + (1 * 8 + 1))

### 2

A base **Iris** (Fisher, 1936; disponível na UCI e embutida no scikit-learn) tem 150 flores, 4 medidas e 3 espécies.

1. Separe 70% treino e 30% teste de forma **estratificada** (`stratify=y`, `random_state=0`).
2. Monte um `Pipeline` com `StandardScaler` e `MLPClassifier` (5 neurônios ocultos, `max_iter=2000`, `random_state=0`).
3. Calcule a acurácia no treino e no teste e exiba a matriz de confusão.

**Pergunta:** por que o `StandardScaler` está dentro do `Pipeline`, em vez de ser aplicado antes da separação dos dados?

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix

X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
modelo = make_pipeline(StandardScaler(),
                       MLPClassifier(hidden_layer_sizes=(5,), max_iter=2000, random_state=0))
modelo.fit(X_tr, y_tr)
print("acurácia treino = %.4f | teste = %.4f" % (modelo.score(X_tr, y_tr), modelo.score(X_te, y_te)))
print(confusion_matrix(y_te, modelo.predict(X_te)))

### 3

Use a rede do Exercício 2 (já treinada) para conferir, na prática, as dimensões do Capítulo 1.

1. Mostre a forma de cada matriz em `coefs_` e de cada vetor em `intercepts_`.
2. Calcule o número total de parâmetros e compare com $P=\sum_l(n_ln_{l-1}+n_l)$.
3. Qual é a função de ativação da saída escolhida pelo scikit-learn para este problema?

**Pergunta:** como as formas de `coefs_` se relacionam com $W^{(l)}$ do livro?

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier

X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
modelo = make_pipeline(StandardScaler(),
                       MLPClassifier(hidden_layer_sizes=(5,), max_iter=2000, random_state=0))
modelo.fit(X_tr, y_tr)
rede = modelo[-1]

print("formas de coefs_:     ", [w.shape for w in rede.coefs_])
print("formas de intercepts_:", [b.shape for b in rede.intercepts_])
P = sum(w.size for w in rede.coefs_) + sum(b.size for b in rede.intercepts_)
camadas = [4, 5, 3]
P_formula = sum(n * n_ant + n for n_ant, n in zip(camadas[:-1], camadas[1:]))
print("P =", P, "| fórmula =", P_formula)
print("ativação da saída:", rede.out_activation_)

### Quanto a arquitetura importa? Explorando larguras de uma camada oculta

**Objetivo.** Investigar como o número de neurônios ocultos afeta o desempenho em uma base real.

**Dados.** Base **Wine** (UCI Machine Learning Repository, embutida no scikit-learn): 178 vinhos, 13 medidas químicas, 3 cultivares.

**Tarefas.**
1. Separe os dados em treino (60%), validação (20%) e teste (20%), de forma estratificada e com sementes fixas.
2. Para cada arquitetura $(2),\,(5),\,(10),\,(20),\,(50)$ e $(10,10)$, treine 5 redes com sementes diferentes (`StandardScaler` ajustado só com o treino) e registre média e desvio da acurácia de **validação**.
3. Escolha a arquitetura pela validação. **Só então** avalie no teste, uma única vez.
4. Redija 8 a 10 linhas: o que você observou? A conclusão vale para qualquer base?

**Para ir além (não testado no livro).** Procure no repositório da UCI e no Kaggle bases tabulares pequenas de classificação (por exemplo, variações do conjunto de vinhos ou de diagnósticos médicos) e repita o procedimento. Confira a descrição, a licença e as colunas antes de usar.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier

X, y = load_wine(return_X_y=True)
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=0,
                                          stratify=y_tmp)
print("tamanhos treino/val/teste:", len(y_tr), len(y_va), len(y_te))

def construir(h, semente):
    return make_pipeline(StandardScaler(),
                         MLPClassifier(hidden_layer_sizes=h, max_iter=3000, random_state=semente))

resultados = []
for h in [(2,), (5,), (10,), (20,), (50,), (10, 10)]:
    acc = [construir(h, s).fit(X_tr, y_tr).score(X_va, y_va) for s in range(5)]
    modelo = construir(h, 4).fit(X_tr, y_tr)
    P = sum(w.size for w in modelo[-1].coefs_) + sum(b.size for b in modelo[-1].intercepts_)
    resultados.append((h, P, np.mean(acc), np.std(acc)))
    print("%-10s P=%4d  acc_val = %.3f ± %.3f" % (str(h), P, np.mean(acc), np.std(acc)))

melhor = max(resultados, key=lambda r: r[2])
print("arquitetura escolhida pela validação:", melhor[0])
final = construir(melhor[0], 0).fit(np.vstack([X_tr, X_va]), np.concatenate([y_tr, y_va]))
print("acurácia no teste (avaliada uma única vez): %.3f" % final.score(X_te, y_te))